In [ ]:
!pip install -q transformers torch peft bitsandbytes accelerate matplotlib

In [ ]:
# ── Configuration ──────────────────────────────────────────
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
BANNED_INGREDIENTS = ["garlic", "butter", "heavy cream", "soy sauce", "sugar"]
NUM_TRAINING_PROMPTS = 300
NUM_EVAL_PROMPTS = 30
NUM_EPOCHS = 3
TEMPERATURE = 0.8
MAX_TOKENS = 1024
WEIGHT_SAVE_PATH = "/content/drive/MyDrive/lora_weights"
FORCE_RETRAIN = False
PROMPT_TEMPLATE = "Write a recipe for {dish}. Include a title, an Ingredients: section listing all ingredients, and step-by-step cooking instructions."

In [ ]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
assert HF_TOKEN, "Set HF_TOKEN via Colab secrets or environment variable."

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Base model loaded: {MODEL_ID}")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from dish_list import DISHES

training_dishes = DISHES[0:300]
eval_dishes = DISHES[300:330]
print(f"Training dishes: {len(training_dishes)} (indices 0-299)")
print(f"Eval dishes: {len(eval_dishes)} (indices 300-329)")

In [ ]:
class PreferenceHead(torch.nn.Module):
    """Scores recipe hidden states → scalar preference score."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.proj = torch.nn.Linear(hidden_dim, 128)
        self.act = torch.nn.GELU()
        self.out = torch.nn.Linear(128, 1)

    def forward(self, hidden_state):
        if hidden_state.ndim == 2:
            h = hidden_state.mean(dim=0)
        else:
            h = hidden_state
        return self.out(self.act(self.proj(h))).squeeze()

hidden_dim = model.config.hidden_size
pref_head = PreferenceHead(hidden_dim).to(model.device)
print(f"PreferenceHead initialized (hidden_dim={hidden_dim})")

In [ ]:
def generate_recipe(dish: str) -> str:
    """Generate a single recipe via one LLM call. Returns plain text."""
    prompt = PROMPT_TEMPLATE.format(dish=dish)
    messages = [{"role": "user", "content": prompt}]
    encoded = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True).to(model.device)
    input_ids = encoded["input_ids"]
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    del encoded, outputs
    torch.cuda.empty_cache()
    return generated

In [ ]:
def detect_banned_ingredients(recipe_text: str, banned: list[str]) -> list[str]:
    """Case-insensitive substring match against full recipe text."""
    text_lower = recipe_text.lower()
    return [ing for ing in banned if ing.lower() in text_lower]

In [ ]:
import os
from peft import PeftModel

def save_weights(model, pref_head, path: str):
    """Save LoRA adapters + preference head state dict."""
    os.makedirs(path, exist_ok=True)
    lora_path = os.path.join(path, "lora")
    model.save_pretrained(lora_path)
    torch.save(pref_head.state_dict(), os.path.join(path, "preference_head.pt"))
    print(f"Weights saved to {path}")

def load_weights(model, pref_head, path: str) -> bool:
    """Load saved weights. Returns True if loaded, False if not found."""
    lora_path = os.path.join(path, "lora")
    pref_path = os.path.join(path, "preference_head.pt")
    if os.path.exists(lora_path) and os.path.exists(pref_path):
        try:
            model.load_adapter(lora_path, adapter_name="default")
            pref_head.load_state_dict(torch.load(pref_path, map_location=model.device))
            return True
        except Exception as e:
            print(f"Warning: Failed to load weights: {e}. Proceeding with training.")
            return False
    return False

In [ ]:
import json as _ckpt_json

CHECKPOINT_PATH = os.path.join(WEIGHT_SAVE_PATH, "checkpoint.json")
CACHE_PATH = os.path.join(WEIGHT_SAVE_PATH, "recipe_cache.json")

skip_training = False
resume_epoch = 0
resume_step = 0
cached_recipes = {}
resume_gradient_updates = 0
resume_llm_calls = 0

if not FORCE_RETRAIN:
    # Check for mid-training checkpoint first
    if os.path.exists(CHECKPOINT_PATH):
        ckpt = _ckpt_json.load(open(CHECKPOINT_PATH))
        if ckpt.get("completed"):
            # Training finished previously — load final weights, skip training
            if load_weights(model, pref_head, WEIGHT_SAVE_PATH):
                print(f"Completed training weights loaded from {WEIGHT_SAVE_PATH}. Training skipped.")
                skip_training = True
        else:
            # Resume from mid-training checkpoint
            resume_epoch = ckpt["epoch"]
            resume_step = ckpt["step"]
            resume_gradient_updates = ckpt.get("gradient_updates", 0)
            resume_llm_calls = ckpt.get("llm_calls", 0)
            if os.path.exists(CACHE_PATH):
                cached_recipes = _ckpt_json.load(open(CACHE_PATH))
            if load_weights(model, pref_head, WEIGHT_SAVE_PATH):
                print(f"Resuming from epoch {resume_epoch+1}, step {resume_step}, "
                      f"{resume_gradient_updates} updates, {resume_llm_calls} LLM calls")
            else:
                print("Checkpoint found but weights failed to load. Restarting training.")
                resume_epoch = 0
                resume_step = 0
                cached_recipes = {}
                resume_gradient_updates = 0
                resume_llm_calls = 0
    elif load_weights(model, pref_head, WEIGHT_SAVE_PATH):
        print(f"Weights loaded from {WEIGHT_SAVE_PATH}. Training skipped.")
        skip_training = True
    else:
        print("No saved weights found. Proceeding with training.")
else:
    print("FORCE_RETRAIN=True. Ignoring saved weights.")

In [ ]:
if not skip_training:
    CKPT_EVERY = 10  # Save checkpoint every N steps during epoch 0 (generation phase)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(pref_head.parameters()),
        lr=1e-4,
    )
    cache = dict(cached_recipes)  # Restore from checkpoint if resuming
    gradient_updates = resume_gradient_updates
    llm_calls = resume_llm_calls

    def _save_checkpoint(epoch, step, completed=False):
        """Save weights + checkpoint metadata + recipe cache to Drive."""
        save_weights(model, pref_head, WEIGHT_SAVE_PATH)
        _ckpt_json.dump(
            {"epoch": epoch, "step": step, "gradient_updates": gradient_updates,
             "llm_calls": llm_calls, "completed": completed},
            open(CHECKPOINT_PATH, "w"),
        )
        _ckpt_json.dump(cache, open(CACHE_PATH, "w"))

    for epoch in range(resume_epoch, NUM_EPOCHS):
        epoch_loss = 0.0
        start_step = resume_step if epoch == resume_epoch else 0

        for i, dish in enumerate(training_dishes):
            if i < start_step:
                continue  # Skip already-completed steps when resuming

            # Generate or reuse cached recipe
            if dish not in cache:
                recipe_text = generate_recipe(dish)
                cache[dish] = recipe_text
                llm_calls += 1
            else:
                recipe_text = cache[dish]

            # Compute penalization signal
            found = detect_banned_ingredients(recipe_text, BANNED_INGREDIENTS)
            signal = 0.0 if found else 1.0

            # Forward pass for hidden states (truncate to 256 tokens to save VRAM)
            inputs = tokenizer(recipe_text, return_tensors="pt", truncation=True, max_length=256)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[-1].squeeze(0).mean(dim=0).float()

            # Preference head prediction + MSE loss
            predicted = pref_head(hidden)
            target = torch.tensor(signal, dtype=torch.float32, device=model.device)
            loss = torch.nn.functional.mse_loss(predicted, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            gradient_updates += 1
            epoch_loss += loss.item()

            # Free memory
            del inputs, outputs, hidden, predicted, target, loss
            torch.cuda.empty_cache()

            if (i + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}/{NUM_EPOCHS}, Step {i+1}/{NUM_TRAINING_PROMPTS}")

            # Checkpoint every CKPT_EVERY steps during generation epoch,
            # every 50 steps during cache-only epochs
            ckpt_interval = CKPT_EVERY if epoch == 0 else 50
            if (i + 1) % ckpt_interval == 0:
                _save_checkpoint(epoch, i + 1)
                print(f"  ✔ Checkpoint saved (epoch {epoch+1}, step {i+1})")

        avg_loss = epoch_loss / max(NUM_TRAINING_PROMPTS - start_step, 1)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} complete. Avg loss: {avg_loss:.4f}")

        # Checkpoint at end of each epoch
        _save_checkpoint(epoch + 1, 0)
        print(f"  ✔ Epoch checkpoint saved")

    # Mark training as completed
    _save_checkpoint(NUM_EPOCHS, 0, completed=True)
    print(f"\nTraining complete. Gradient updates: {gradient_updates}, LLM calls: {llm_calls}")

In [ ]:
if not skip_training:
    print(f"Final weights at {WEIGHT_SAVE_PATH}")
    # Clean up checkpoint files now that training is complete
    if os.path.exists(CACHE_PATH):
        os.remove(CACHE_PATH)
        print("Recipe cache cleaned up.")

In [ ]:
eval_results = []
for i, dish in enumerate(eval_dishes):
    print(f"[{i+1}/{NUM_EVAL_PROMPTS}] Generating recipe for: {dish}")
    recipe_text = generate_recipe(dish)
    banned_found = detect_banned_ingredients(recipe_text, BANNED_INGREDIENTS)
    eval_results.append({
        "dish": dish,
        "recipe_text": recipe_text,
        "banned_found": banned_found,
        "contains_banned": len(banned_found) > 0,
    })
    print(f"  Banned found: {banned_found if banned_found else 'None'}")
print(f"\nEvaluation complete. {len(eval_results)} recipes generated.")

In [ ]:
lora_per_ingredient = {ing: 0 for ing in BANNED_INGREDIENTS}
lora_with_banned = 0
for r in eval_results:
    if r["banned_found"]:
        lora_with_banned += 1
    for ing in r["banned_found"]:
        if ing in lora_per_ingredient:
            lora_per_ingredient[ing] += 1

lora_clean = len(eval_results) - lora_with_banned
lora_summary = {
    "total_recipes": len(eval_results),
    "recipes_with_banned": lora_with_banned,
    "recipes_clean": lora_clean,
    "per_ingredient_count": lora_per_ingredient,
}
print(f"LoRA - Recipes with banned ingredients: {lora_with_banned}/{len(eval_results)}")
print(f"LoRA - Clean recipes: {lora_clean}/{len(eval_results)}")

In [ ]:
import json
from datetime import datetime

lora_output = {
    "metadata": {
        "notebook": "lora",
        "model_id": MODEL_ID,
        "banned_ingredients": BANNED_INGREDIENTS,
        "num_eval_prompts": NUM_EVAL_PROMPTS,
        "timestamp": datetime.now().isoformat(),
    },
    "recipes": eval_results,
    "summary": lora_summary,
}

with open("lora_results.json", "w") as f:
    json.dump(lora_output, f, indent=2)
print("LoRA results saved to lora_results.json")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

percentages = [lora_per_ingredient[ing] / len(eval_results) * 100 for ing in BANNED_INGREDIENTS]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(BANNED_INGREDIENTS))
bars = ax.bar(x, percentages, color="#DD8452", width=0.5)
ax.set_xlabel("Banned Ingredient")
ax.set_ylabel("% of Recipes Containing Ingredient")
ax.set_title("LoRA: Banned Ingredient Frequency")
ax.set_xticks(x)
ax.set_xticklabels(BANNED_INGREDIENTS, rotation=45, ha="right")
ax.set_ylim(0, 105)
ax.bar_label(bars, fmt="%.0f%%", padding=3)
fig.tight_layout()
plt.show()